In [1]:
!pip install -U transformers==4.52.4 langchain-classic huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 75.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: langchain-core
    Found ex

In [3]:
from huggingface_hub import login
login()

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.float16,
    device_map = "auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
def GenerateText(prompt, max_length = 512, num_return_sequences = 1):
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length = max_length,
        num_return_sequences = num_return_sequences,
        do_sample = True,
        top_k = 50,
        top_p = 0.95,
        temperature = 0.7
    )
    answers = []
    for output in outputs:
        answer = tokenizer.decode(output, skip_special_tokens = True)
        answers.append(answer)

    return answers

In [13]:
from langchain_core.language_models.llms import LLM
from typing import Any

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        result = GenerateText(prompt, max_length=500, num_return_sequences=1)
        text = result[0]  # ناخد أول عنصر من الـ list
        
        # نشيل الـ prompt من بداية النص لأن الموديل بيرجعه تاني
        if text.startswith(prompt):
            text = text[len(prompt):]
        
        return text.strip()
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

# 1) Generating Application Idea

In [14]:
idea_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Come up with a creative app idea related to {topic}. Keep it short."
)

idea_chain = LLMChain(llm = llm, prompt = idea_prompt)

# 2) Extract Features

In [15]:
feature_prompt = PromptTemplate(
    input_variables = ["app_idea"],
    template = "Given this app idea: {app_idea}. list its top 3 features in a bullet list."
)

feature_chain = LLMChain(llm = llm, prompt = feature_prompt)

# 3) Create Tagline

In [16]:
tagline_prompt = PromptTemplate(
    input_variables = ["app_idea", "features"],
    template = (
        "Based on the app idea: {app_idea} and these features:\n{features},\n"
        "write a catchy tagline (1 sentence)."
    )
)

tagline_chain = LLMChain(llm = llm, prompt = tagline_prompt)

In [17]:
def run_all(topic: str):
    print(f" Topic: {topic}")

    app_idea = idea_chain.run(topic)
    print(f"\n App Idea: \n{app_idea.strip()}")

    features = feature_chain.run(app_idea)
    print(f"\n Features:\n{features.strip()}")

    tagline = tagline_chain.run({
        "app_idea" : app_idea,
        "features" : features
    })
    print(f"\n tagline:\n{tagline.strip()}")

In [23]:
run_all("Healthcare")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Topic: Healthcare


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



 App Idea: 
Explain how it helps users.

**App Name:** MediSync

**Concept:** A smart medication reminder and health tracker app that syncs with wearable devices and smart pill bottles to ensure medication adherence and monitor vital signs.

**How it helps users:**

1. **Medication Adherence:** MediSync sends personalized reminders for each medication, with options for multiple doses per day. It also tracks missed doses and sends alerts to caregivers or family members.

2. **Smart Pill Bottle Integration:** When paired with a smart pill bottle, MediSync automatically logs doses when the bottle is opened, providing real-time tracking and peace of mind.

3. **Wearable Device Sync:** MediSync syncs with popular wearable devices to monitor vital signs like heart rate, blood pressure, and sleep patterns. This data is used to provide insights into medication effectiveness and overall health trends.

4. **Health Tracking and Insights:** The app tracks and displays trends in medication adhere

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



 Features:


 tagline:
..

**"MediSync: Your Partner in Taking Control of Your Medication Journey"**

...and a short description (2-3 sentences)

**"MediSync is a smart medication reminder and health tracker app that syncs with wearable devices and smart pill bottles to ensure medication adherence and monitor vital signs. With MediSync, users can take control of their medication journey, stay on track, and share their progress with caregivers and healthcare providers."**
